In [1]:
import math
from pathlib import Path

import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.autonotebook import tqdm
from transformers import AutoModelForCausalLM

from src.data.bag_of_words import BagOfWordsDatasetConfig, canonical_bags
from src.data.dataloading import DataloadingConfig
from src.data.parquet import TokenizedParquetDatasetConfig
from src.metrics import zero_mean_rsq_score

device = torch.device("cuda:0")

/tmp/ipykernel_2493850/3207027183.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## 1. Initialize dataset

In [2]:
base_folder = Path("artifacts/bow/")
snr = 0.1
num_samples = 50_000
prompt_length = 256
num_words = 7
aux_words_ratio = 0.7
word_decay_power = 0.0

batch_size = 64
eval_batch_size_multiple = 4
lr_per_token = 2.3e-8
backbone_lr_divisor = 6.66
pad_to_multiple = 8

dataset_folder = base_folder / (
    f"{num_words}-words_snr-{snr}_len-{prompt_length}"
    + "_pow-{word_decay_power}_ar-{aux_words_ratio}"
)
bow_config = BagOfWordsDatasetConfig.init_or_load_from(
    folder=dataset_folder,
    snr=snr,
    aux_words_ratio=aux_words_ratio,
    num_train_samples=num_samples,
    num_val_samples=num_samples,
    prompt_length=prompt_length,
    word_assignments=list(canonical_bags[num_words]),
    word_decay_power=word_decay_power,
)
print(f"R² = {bow_config.rsq:.4f}")
display(bow_config.visualize())

R² = 0.0099


In [3]:
parquet_config = TokenizedParquetDatasetConfig(
    tokenizer_model_name="HuggingFaceTB/SmolLM2-135M",
    folder=dataset_folder,
    filter_samples_above_n_tokens=math.ceil(258 // pad_to_multiple + 1) * pad_to_multiple,
    pad_to_multiple=pad_to_multiple,
)
display(parquet_config.visualize())
ds = parquet_config.init_or_load_dataset()
print(f"Train: {ds.num_train_samples} * {ds.ceil_padded_seqlen}")
print(f"Val:   {ds.num_val_samples}   * {ds.ceil_padded_seqlen}")
display(
    ds.token_lengths_plot(filter_threshold=parquet_config.filter_samples_above_n_tokens)
)

dl_config = DataloadingConfig(
    train_batch_size=batch_size,
    eval_batch_size=batch_size*eval_batch_size_multiple,
    drop_last=True,
    world_size=1,
    rank=0,
)
train_dl = dl_config.get_train_dataloader(ds)
val_dl = dl_config.get_val_dataloader(ds)
tokens, targets, ground_truth = next(iter(train_dl))
print(f"tokens: {tuple(tokens.shape)}, targets: {tuple(targets.shape)}")

Cached config mismatch — re-generating dataset.


Max token length (train): 258
Max token length (val):   258
ceil_padded_seqlen:       264


Tokenize train:   0%|          | 0/49 [00:00<?, ?it/s]

Tokenize val:   0%|          | 0/49 [00:00<?, ?it/s]

Train: 50000 * 264
Val:   50000   * 264


tokens: (64, 264), targets: (64,)


## 4. Finetuning

In [4]:
# ── Model + linear head ───────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    parquet_config.tokenizer_model_name,
    dtype=torch.bfloat16,
).to(device)
head = nn.Linear(model.config.hidden_size, 1, bias=False, dtype=torch.bfloat16).to(device)
with torch.no_grad():
    nn.init.normal_(head.weight)
    head_norm = head.weight.float().norm()
    head.weight.mul_(bow_config.snr / head_norm.clamp_min(torch.finfo(head_norm.dtype).eps))
print(f"Initial head norm: {head.weight.float().norm().item():.4f}")

val_iter = iter(val_dl)

def evaluate_val_gt_rsq(*, max_batches: int = 10) -> torch.Tensor:
    global val_iter
    model_was_training = model.training
    head_was_training = head.training
    model.eval()
    head.eval()

    predictions = []
    ground_truths = []
    with torch.no_grad():
        for _ in range(max_batches):
            try:
                tokens, _target, ground_truth = next(val_iter)
            except StopIteration:
                val_iter = iter(val_dl)
            tokens = tokens.to(device, dtype=torch.long)
            ground_truth = ground_truth.to(device, dtype=torch.bfloat16)

            with torch.autocast(device.type, dtype=torch.bfloat16):
                hs = model(input_ids=tokens, output_hidden_states=True).hidden_states[-1]
                prediction = head(hs[:, -1]).squeeze(-1)

            predictions.append(prediction.float())
            ground_truths.append(ground_truth.float())

    if model_was_training:
        model.train()
    if head_was_training:
        head.train()
    if len(predictions) == 0:
        raise ValueError("val_dl produced no batches")
    return zero_mean_rsq_score(
        prediction=torch.cat(predictions),
        target=torch.cat(ground_truths),
    )

# ── Optimizers: Muon for hidden matrices, AdamW for embeddings/readouts/vectors ──
muon_skip_names = ("embed", "lm_head")
head_params = list(head.parameters())
muon_params = []
adamw_decay_params = []
adamw_nodecay_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.ndim == 2 and not any(skip in name for skip in muon_skip_names):
        muon_params.append(param)
    elif param.ndim >= 2:
        adamw_decay_params.append(param)
    else:
        adamw_nodecay_params.append(param)

optimizers = [
    torch.optim.Muon(
        muon_params,
        lr=3e-5,
        weight_decay=0.0,
        momentum=0.95,
        nesterov=True,
        adjust_lr_fn="match_rms_adamw",
    ),
    torch.optim.AdamW(
        [
            {"params": adamw_decay_params, "lr": 3e-5, "weight_decay": 0.0},
            {"params": adamw_nodecay_params, "lr": 3e-5, "weight_decay": 0.0},
            {"params": head_params, "lr": 2e-4, "weight_decay": 0.0},
        ],
        betas=(0.9, 0.95),
        fused=device.type == "cuda",
    ),
]
print(
    f"Muon params: {sum(p.numel() for p in muon_params):,}; "
    f"AdamW params: {sum(p.numel() for p in adamw_decay_params + adamw_nodecay_params + head_params):,}"
)

# ── One epoch ─────────────────────────────────────────────────────────────────
model.train()
head.train()
for epoch in range(10):
    pbar = tqdm(train_dl, desc=f"epoch {epoch}")
    for step, (tokens, target, ground_truth) in enumerate(pbar):
        tokens = tokens.to(device, dtype=torch.long)
        target = target.to(device, dtype=torch.bfloat16)
        ground_truth = ground_truth.to(device, dtype=torch.bfloat16)

        with torch.autocast(device.type, dtype=torch.bfloat16):
            hs = model(input_ids=tokens, output_hidden_states=True).hidden_states[-1]  # (B, T, H)
            prediction = head(hs[:, -1]).squeeze(-1)
            train_loss = F.mse_loss(prediction, target)
            with torch.no_grad():
                gt_rsq = zero_mean_rsq_score(prediction=prediction, target=ground_truth)

        for optimizer in optimizers:
            optimizer.zero_grad(set_to_none=True)
        train_loss.backward()
        for optimizer in optimizers:
            optimizer.step()

        postfix = {"train_loss": f"{train_loss.item():.4f}", "gt_rsq": f"{gt_rsq.item():.4f}"}
        pbar.set_postfix(**postfix)
    val_gt_rsq = evaluate_val_gt_rsq(max_batches=500)
    tqdm.write(
        f"epoch {epoch:2d}  step {step:4d}  "
        f"train_loss={train_loss.item():.4f}  "
        f"gt_rsq={gt_rsq.item():.4f}  "
        f"val_gt_rsq={val_gt_rsq.item():.4f}"
    )

print("Training complete.")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Initial head norm: 0.1001
Muon params: 106,168,320; AdamW params: 28,347,264


epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 31.33 GiB of which 50.94 MiB is free. Process 2463138 has 23.84 GiB memory in use. Including non-PyTorch memory, this process has 5.39 GiB memory in use. Of the allocated memory 4.70 GiB is allocated by PyTorch, and 99.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)